# RDataFrame: systematic variations

This notebook introduces the experimental RDataFrame API for handling systematic variations.

The goal is not to cover every possible systematic uncertainty used in a full physics analysis. Instead, the goal is to understand the **computational pattern**:

> a nominal analysis produces one result; an analysis with systematic variations produces a family of results, one for each varied universe.

In a traditional event-loop analysis this often means duplicating code or manually filling many histograms. RDataFrame provides a way to express these variations in the computation graph.

**Learning goals**

By the end of this notebook you should understand:

- what a systematic variation means computationally,
- how to register a variation with `Vary`,
- how variations propagate through `Define`, `Filter`, and actions,
- how to retrieve nominal and varied results with `VariationsFor`,
- how to vary several columns together in lockstep.

This is an advanced topic. It is best read after the notebooks on RDataFrame basics and collections.


## Before we start: nominal and varied universes

Suppose an analysis fills a histogram of transverse momentum, `pt`.

For the nominal analysis we use the nominal value of `pt`.

For a systematic uncertainty, we may want to repeat the same analysis with:

- `pt_down = 0.95 * pt`,
- `pt_up   = 1.05 * pt`.

Conceptually, we want the same analysis graph to run in three universes:

```text
nominal universe:   pt
variation down:     0.95 * pt
variation up:       1.05 * pt
```

The important point is that we do **not** want to duplicate the full analysis manually.


## Setup

We use the same small toy dataset as in the collections tutorial. It contains event-level arrays such as `px`, `py`, `E`, etc.

The variable names are generic, but the pattern is the same as in a real analysis: define observables, apply selections, and fill histograms.


In [ ]:
import ROOT

# Input dataset used by the official ROOT tutorials.
treename = "Events"
filename = "data/particles.root"

df = ROOT.RDataFrame(treename, filename)


## Build the nominal analysis first

Before adding systematic variations, it is useful to write the nominal analysis.

Here we compute

```cpp
pt = sqrt(px*px + py*py)
```

then select objects with `E > 100`, and finally fill a histogram.

This is the analysis we will later run in several varied universes.


In [ ]:
nominal_pt = (
    df
    .Define("pt", "sqrt(px*px + py*py)")
    .Define("good_pt", "pt[E > 100]")
    .Histo1D(("h_pt_nominal", "Selected p_{T};p_{T};Events", 40, 0, 200), "good_pt")
)

%jsroot on
canvas = ROOT.TCanvas()
nominal_pt.Draw()
canvas.Draw()


## Registering variations for one column

Now we register two variations for the column `pt`.

The idea is:

```text
pt nominal  -> pt
pt down     -> 0.95 * pt
pt up       -> 1.05 * pt
```

The `Vary` call tells RDataFrame how to produce the varied versions of a column. The rest of the analysis can be written once.

A few important details:

- `colName="pt"` says which column is varied.
- `expression=...` returns the varied values.
- `variationTags=["down", "up"]` gives human-readable labels.
- `variationName="pt_scale"` controls the name used later when retrieving the results.

The variations automatically propagate to columns derived from `pt`, such as `good_pt`.


In [ ]:
df_with_pt = df.Define("pt", "sqrt(px*px + py*py)")

h_pt = (
    df_with_pt
    .Vary(
        colName="pt",
        expression="ROOT::RVec<ROOT::RVecD>{pt * 0.95, pt * 1.05}",
        variationTags=["down", "up"],
        variationName="pt_scale",
    )
    .Define("good_pt", "pt[E > 100]")
    .Histo1D(("h_pt", "Selected p_{T};p_{T} (GeV);Events", 40, 0, 200), "good_pt")
)


## Retrieving nominal and varied histograms

The action `h_pt` represents the nominal histogram. To retrieve also the varied histograms, use:

```python
ROOT.RDF.Experimental.VariationsFor(h_pt)
```

In [ ]:
all_histos = ROOT.RDF.Experimental.VariationsFor(h_pt)

In [ ]:
%jsroot on
canvas = ROOT.TCanvas()

all_histos["nominal"].SetLineColor(ROOT.kBlack)
all_histos["nominal"].SetLineWidth(2)
all_histos["nominal"].Draw("HIST")

all_histos["pt_scale:down"].SetLineColor(ROOT.kRed + 1)
all_histos["pt_scale:down"].SetLineWidth(2)
all_histos["pt_scale:down"].Draw("HIST SAME")

all_histos["pt_scale:up"].SetLineColor(ROOT.kBlue + 1)
all_histos["pt_scale:up"].SetLineWidth(2)
all_histos["pt_scale:up"].Draw("HIST SAME")

legend = ROOT.TLegend(0.60, 0.70, 0.88, 0.88)
legend.AddEntry(all_histos["nominal"], "nominal", "l")
legend.AddEntry(all_histos["pt_scale:down"], "pt scale down", "l")
legend.AddEntry(all_histos["pt_scale:up"], "pt scale up", "l")
legend.Draw()

canvas.Draw()


### What happened?

The analysis code after `Vary` was written only once:

```python
.Define("good_pt", "pt[E > 100]")
.Histo1D(..., "good_pt")
```

but RDataFrame evaluated it for the nominal and varied values of `pt`.

This is the main reason this feature is useful: the analysis graph remains compact even when several systematic variations are included.


## Registering variations for multiple columns simultaneously

The `Vary` function also allows to vary multiple columns simultaneously (in "lockstep"). The expression in this case must return an RVec of RVecs, one per column: each inner vector contains the varied values for one column, and the inner vectors follow the same ordering as the column names passed as first argument. Besides the variation tags, in this case we also have to explicitly pass a variation name as there is no one column name that can be used as default.

In [ ]:
df2 = ROOT.RDataFrame(treename, filename)

h_pt_lockstep = (
    df2
    .Vary(
        colNames=["px", "py"],
        expression="ROOT::RVec<ROOT::RVec<ROOT::RVecD>>{{px * 0.95, px * 1.05}, {py * 0.95, py * 1.05}}",
        variationTags=["down", "up"],
        variationName="momentum_scale",
    )
    .Define("pt", "sqrt(px*px + py*py)")
    .Define("good_pt", "pt[E > 100]")
    .Histo1D(("h_pt_lockstep", "Selected p_{T};p_{T} (GeV);Events", 40, 0, 200), "good_pt")
)


In [ ]:
all_histos_lockstep = ROOT.RDF.Experimental.VariationsFor(h_pt_lockstep)

In [ ]:
%jsroot on
canvas = ROOT.TCanvas()

all_histos_lockstep["nominal"].SetLineColor(ROOT.kBlack)
all_histos_lockstep["nominal"].SetLineWidth(2)
all_histos_lockstep["nominal"].Draw("HIST")

all_histos_lockstep["momentum_scale:down"].SetLineColor(ROOT.kRed + 1)
all_histos_lockstep["momentum_scale:down"].SetLineWidth(2)
all_histos_lockstep["momentum_scale:down"].Draw("HIST SAME")

all_histos_lockstep["momentum_scale:up"].SetLineColor(ROOT.kBlue + 1)
all_histos_lockstep["momentum_scale:up"].SetLineWidth(2)
all_histos_lockstep["momentum_scale:up"].Draw("HIST SAME")

legend = ROOT.TLegend(0.55, 0.70, 0.88, 0.88)
legend.AddEntry(all_histos_lockstep["nominal"], "nominal", "l")
legend.AddEntry(all_histos_lockstep["momentum_scale:down"], "px,py scale down", "l")
legend.AddEntry(all_histos_lockstep["momentum_scale:up"], "px,py scale up", "l")
legend.Draw()

canvas.Draw()


## Connection to a real analysis

In a real particle-physics analysis, the same pattern appears for many uncertainties:

- energy or momentum scale variations,
- resolution variations,
- calibration variations,
- object-level correction variations.

The computational idea is always similar:

```text
nominal input columns -> varied input columns -> same analysis selections and definitions -> nominal and varied histograms
```

This notebook showed the minimal mechanism. A full analysis framework will usually add bookkeeping, naming conventions, output files, and combinations of many systematic sources.
